## 5.1 KPI Catalogue

### A. Overall Business Performance

| KPI | Definition | Formula |
|---|---|---|
| Total Rentals | Total number of bike rentals | SUM(cnt) |
| Average Hourly Rentals | Average rentals per hourly observation | AVG(cnt) |
| Total Casual Rentals | Total rentals from casual customers | SUM(casual) |
| Total Registered Rentals | Total rentals from registered customers | SUM(registered) |

### B. Customer Analytics

| KPI | Definition | Formula |
|---|---|---|
| Registered Customer Share | Percentage of rentals from registered customers | SUM(registered) / SUM(cnt) * 100 |
| Casual Customer Share | Percentage of rentals from casual customers | SUM(casual) / SUM(cnt) * 100 |
| Casual-to-Registered Ratio | Relative rental contribution of casual customers | SUM(casual) / SUM(registered) |

### C. Demand Management

| KPI | Definition | Formula |
|---|---|---|
| Peak Demand Hour | Hour with highest average rentals | ARGMAX(AVG(cnt) by hour) |
| Lowest Demand Hour | Hour with lowest average rentals | ARGMIN(AVG(cnt) by hour) |
| Peak-to-Off-Peak Ratio | Ratio between highest and lowest average hourly demand | MAX(hourly AVG(cnt)) / MIN(hourly AVG(cnt)) |

### D. Temporal Analytics

| KPI | Definition | Formula |
|---|---|---|
| Average Weekday Demand | Average rentals per weekday observation | AVG(cnt) where isweekend = 0 |
| Average Weekend Demand | Average rentals per weekend observation | AVG(cnt) where isweekend = 1 |
| Weekend Demand Index | Weekend demand relative to weekday demand | Weekend AVG(cnt) / Weekday AVG(cnt) * 100 |
| Seasonal Demand | Average rentals grouped by season | AVG(cnt) by season |

### E. Environmental Analytics

| KPI | Definition | Formula |
|---|---|---|
| Clear Weather Demand | Average rentals during clear weather | AVG(cnt) where weathersit = 1 |
| Light Rain/Snow Demand | Average rentals during light rain/snow | AVG(cnt) where weathersit = 3 |
| Weather Demand Difference | Difference in average rentals between clear and light rain/snow | Clear AVG(cnt) - Light Rain/Snow AVG(cnt) |
| Weather Demand Reduction | Reduction in demand due to bad weather | Weather Demand Reduction (%) = (Clear Average - Light Rain/Snow Average) / Clear Average * 100

| Executive KPIs | Operational KPIs |
|---|---|
| Total Rentals | Peak Demand Hour |
| Average Rentals per Hourly Record | Lowest Demand Hour |
| Registered Customer Share | Peak-to-Off-Peak Ratio |
| Casual Customer Share | Average Weekday/Weekend Demand |
| Seasonal Demand | Weather Demand Reduction |

## 5.2 Overall Business Performance KPIs

Calculate the core business performance metrics using the transformed dataset.

The metrics provide a high-level view of rental volume and customer contribution.

In [1]:
import pandas as pd
from pathlib import Path

ROOT_DIR = Path.cwd().parent

TRANSFORMED_DATA_PATH = ROOT_DIR / "data" / "processed" / "hourly_transformed.parquet"

business_data = pd.read_parquet(TRANSFORMED_DATA_PATH)

business_data.sample(5)

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,...,day_of_week,day_name,quarter,isweekend,season_name,weather_name,temp_c,atemp_c,humidity_pct,windspeed_kmh
14450,14451,2012-08-30,3,1,8,7,0,4,1,1,...,3,Thursday,3,0,Fall,Clear,24.96,30.305,73.0,0.0000
15850,15851,2012-10-27,4,1,10,15,0,6,0,2,...,5,Saturday,4,1,Winter,Mist/Cloudy,21.84,26.515,60.0,19.0012
8745,8746,2012-01-05,1,1,1,5,0,4,1,2,...,3,Thursday,1,0,Spring,Mist/Cloudy,7.80,11.365,59.0,6.0032
13727,13728,2012-07-31,3,1,7,4,0,2,1,1,...,1,Tuesday,3,0,Fall,Clear,25.74,30.305,83.0,6.0032
10498,10499,2012-03-18,1,1,3,13,0,0,0,2,...,6,Sunday,1,1,Spring,Mist/Cloudy,17.94,22.725,82.0,11.0014


In [2]:
business_data[['cnt','casual', 'registered']].describe()

,cnt,casual,registered
count,17379.000000,17379.000000,17379.000000
mean,189.463088,35.676218,153.786869
std,181.387599,49.305030,151.357286
min,1.000000,0.000000,0.000000
25%,40.000000,4.000000,34.000000
50%,142.000000,17.000000,115.000000
75%,281.000000,48.000000,220.000000
max,977.000000,367.000000,886.000000


### 5.2.1 Overall Business Performance

Calculate total rental volume, average hourly demand and customer-segment rental volumes.

In [3]:
total_rentals = business_data['cnt'].sum()

average_hourly_rentals = business_data['cnt'].mean()

total_casual_rentals = business_data['casual'].sum()

total_registered_rentals = business_data['registered'].sum()

overall_kpis = pd.DataFrame({
    'KPI':[
        'Total Rentals',
        'Average Hourly Rentals',
        'Total Casual Records',
        'Total Registered Records'
    ],
    'Value':[
        total_rentals,
        round(average_hourly_rentals,2),
        total_casual_rentals,
        total_registered_rentals
    ]
})

overall_kpis

,KPI,Value
0,Total Rentals,3292679.00
1,Average Hourly Rentals,189.46
2,Total Casual Records,620017.00
3,Total Registered Records,2672662.00


In [4]:
assert (
    total_rentals ==
    total_casual_rentals + total_registered_rentals
)

assert len(business_data) == 17379

print("Total rental consistency:", total_rentals == (
    total_casual_rentals + total_registered_rentals
))

print("Observation count:", len(business_data))

Total rental consistency: True
Observation count: 17379


### 5.2.2 Customer Contribution KPIs

Evaluate the contribution of casual and registered customers to total rental volume.

The objective is to understand customer composition and the relative importance of each segment.

In [5]:
registered_share = (
    business_data['registered'].sum()/business_data['cnt'].sum()
) * 100

casual_share = (
    business_data['casual'].sum()/business_data['cnt'].sum()
) * 100

casual_registered_ratio = (
    business_data['casual'].sum()/business_data['registered'].sum()
) * 100

customer_kpis = pd.DataFrame({
    'KPI':[
        "Registered Customer Share",
        "Casual Customer Share",
        "Casual to Registered Ratio"
    ],
    "Value":[
        registered_share,
        casual_share,
        casual_registered_ratio
    ]
})

customer_kpis

,KPI,Value
0,Registered Customer Share,81.169832
1,Casual Customer Share,18.830168
2,Casual to Registered Ratio,23.198482


In [6]:
assert round(registered_share + casual_share, 2) == 100.00

print("Customer contribution validation: Passed")
print("Registered + Casual share:", round(
    registered_share + casual_share, 2
), "%")

Customer contribution validation: Passed
Registered + Casual share: 100.0 %


### 5.2.3 Demand Management KPIs

Evaluate hourly demand patterns to identify peak and off-peak periods.

The objective is to support operational planning, fleet availability and resource allocation.

In [7]:
hourly_demand = (
    business_data.groupby('hour')['cnt'].mean().reset_index(name = 'average_rentals')
)

peak_hour = hourly_demand.loc[hourly_demand['average_rentals'].idxmax()]

lowest_hour = hourly_demand.loc[hourly_demand['average_rentals'].idxmin()]

peak_offpeak_ratio = (
    peak_hour['average_rentals']/lowest_hour['average_rentals']
)

demand_kpis = pd.DataFrame({
    'KPI':[
        "Peak Hour",
        "Lowest Hour",
        "Peak Offpeak Ratio"
    ],
    'Value':[
        f"{int(peak_hour['hour']):02d}:00",
        f"{int(lowest_hour['hour']):02d}:00",
        peak_offpeak_ratio
    ]
})

demand_kpis

,KPI,Value
0,Peak Hour,17:00
1,Lowest Hour,04:00
2,Peak Offpeak Ratio,72.635972


In [8]:
assert len(hourly_demand) == 24

assert hourly_demand['average_rentals'].notna().all()

assert peak_hour['average_rentals'] >= lowest_hour['average_rentals']

assert peak_offpeak_ratio >= 1

print("Demand KPI validation: Passed")
print("Number of hourly categories:", len(hourly_demand))

Demand KPI validation: Passed
Number of hourly categories: 24


### 5.2.4 Temporal Analytics KPIs

Analyze rental demand across weekdays, weekends and seasons.

The objective is to identify temporal variations that can support operational scheduling and seasonal resource planning.

In [9]:
day_type_demand = (
    business_data.groupby('isweekend')['cnt']
    .mean()
)

weekday_demand = day_type_demand.loc[0]
weekend_demand = day_type_demand.loc[1]

weekend_demand_index = (
    weekend_demand / weekday_demand
) * 100

temporal_kpis = pd.DataFrame({
    'KPI': [
        'Average Weekday Demand',
        'Average Weekend Demand',
        'Weekend Demand Index'
    ],
    'Value': [
        round(weekday_demand, 2),
        round(weekend_demand, 2),
        round(weekend_demand_index, 2)
    ]
})

temporal_kpis

,KPI,Value
0,Average Weekday Demand,191.74
1,Average Weekend Demand,183.85
2,Weekend Demand Index,95.89


In [10]:
seasonal_demand = (
    business_data.groupby('season_name')['cnt']
    .mean()
    .sort_values(ascending=False)
    .reset_index(name='average_rentals')
)

seasonal_demand

,season_name,average_rentals
0,Fall,236.016237
1,Summer,208.344069
2,Winter,198.868856
3,Spring,111.114569


In [11]:
assert len(day_type_demand) == 2
assert len(seasonal_demand) == 4
assert temporal_kpis['Value'].notna().all()
assert seasonal_demand['average_rentals'].notna().all()

print("Temporal KPI validation: Passed")

Temporal KPI validation: Passed


### 5.2.5 Environmental Analytics KPIs

Evaluate rental demand under different weather conditions.

The objective is to understand how weather conditions are associated with rental demand and identify opportunities for weather-informed operational planning.

In [12]:
# Average demand under clear weather
clear_weather_demand = (
    business_data.loc[
        business_data['weathersit'] == 1, 'cnt'
    ].mean()
)

# Average demand under light rain/snow
light_rain_demand = (
    business_data.loc[
        business_data['weathersit'] == 3, 'cnt'
    ].mean()
)

# Absolute demand difference
weather_demand_difference = (
    clear_weather_demand - light_rain_demand
)

# Percentage demand reduction
weather_demand_reduction = (
    weather_demand_difference / clear_weather_demand
) * 100

weather_kpis = pd.DataFrame({
    'KPI': [
        'Clear Weather Demand',
        'Light Rain/Snow Demand',
        'Weather Demand Difference',
        'Weather Demand Reduction (%)'
    ],
    'Value': [
        round(clear_weather_demand, 2),
        round(light_rain_demand, 2),
        round(weather_demand_difference, 2),
        round(weather_demand_reduction, 2)
    ]
})

weather_kpis

,KPI,Value
0,Clear Weather Demand,204.87
1,Light Rain/Snow Demand,111.58
2,Weather Demand Difference,93.29
3,Weather Demand Reduction (%),45.54


In [13]:
assert clear_weather_demand > 0
assert light_rain_demand >= 0
assert weather_demand_difference >= 0

assert 0 <= weather_demand_reduction <= 100

print("Environmental KPI validation: Passed")

Environmental KPI validation: Passed


## 5.3 KPI Consolidation & Validation

Consolidate the calculated business KPIs into a unified report.

The objective is to establish a consistent, validated set of business metrics that can later be reproduced using SQL and Power BI.

In [ ]:
# Convert seasonal results into the common KPI format
customer_kpis = customer_kpis.rename(
    columns={'Values': 'Value'}
)

demand_kpis = demand_kpis.rename(
    columns={'Values': 'Value'}
)

seasonal_kpis = pd.DataFrame({
    'KPI': [
        f"Average Rentals - {row['season_name']}"
        for _, row in seasonal_demand.iterrows()
    ],
    'Value': [
        round(value, 2)
        for value in seasonal_demand['average_rentals']
    ]
})

# Consolidate all KPI groups
kpi_report = pd.concat([
    overall_kpis.assign(Category='Overall Performance'),
    customer_kpis.assign(Category='Customer Analytics'),
    demand_kpis.assign(Category='Demand Management'),
    temporal_kpis.assign(Category='Temporal Analytics'),
    seasonal_kpis.assign(Category='Seasonal Analytics'),
    weather_kpis.assign(Category='Environmental Analytics')
], ignore_index=True)

# Organize columns
kpi_report = kpi_report[
    ['Category', 'KPI', 'Value']
]

kpi_report

,Category,KPI,Value
0,Overall Performance,Total Rentals,3292679.0
1,Overall Performance,Average Hourly Rentals,189.46
2,Overall Performance,Total Casual Records,620017.0
3,Overall Performance,Total Registered Records,2672662.0
4,Customer Analytics,Registered Customer Share,81.169832
5,Customer Analytics,Casual Customer Share,18.830168
6,Customer Analytics,Casual to Registered Ratio,23.198482
7,Demand Management,Peak Hour,17:00
8,Demand Management,Lowest Hour,04:00
9,Demand Management,Peak Offpeak Ratio,72.635972


In [15]:
# Check for missing KPI values
assert kpi_report['Value'].notna().all()

# Check for duplicate KPI names
assert kpi_report['KPI'].is_unique

# Check customer contribution
assert round(
    registered_share + casual_share, 2
) == 100.00

# Check rental consistency
assert total_rentals == (
    total_casual_rentals + total_registered_rentals
)

# Check seasonal coverage
assert len(seasonal_demand) == 4

# Check hourly coverage
assert len(hourly_demand) == 24

print("Consolidated KPI validation: PASSED")
print("Total KPIs:", len(kpi_report))
print("KPI categories:", kpi_report['Category'].nunique())

Consolidated KPI validation: PASSED
Total KPIs: 21
KPI categories: 6


In [16]:
OUTPUT_PATH = (
    ROOT_DIR / "reports" / "business_kpi_report.csv"
)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

kpi_report.to_csv(OUTPUT_PATH, index=False)

print("KPI report exported to:", OUTPUT_PATH)

KPI report exported to: /home/global-rex/Documents/bikeshare-intelligence/reports/business_kpi_report.csv


## 5.4 Business Interpretation & Recommendations

Translate the calculated KPIs into actionable business insights.

The objective is to connect analytical findings with operational and strategic decisions.

In [17]:
business_insights = pd.DataFrame({
    'Finding': [
        'Registered customers contribute 81.17% of rental volume.',
        'Peak demand occurs at 17:00.',
        'Lowest demand occurs at 04:00.',
        'Weekend demand is 95.89% of weekday demand.',
        'Fall has the highest average seasonal demand.',
        'Spring has the lowest average seasonal demand.',
        'Light rain/snow demand is 45.54% lower than clear-weather demand.'
    ],
    'Business Implication': [
        'Registered customers are the primary contributors to rental volume.',
        'Evening operations require adequate bike availability.',
        'Overnight demand is relatively low, potentially providing a maintenance window.',
        'Weekday and weekend demand profiles require different operational planning.',
        'Fleet availability should account for higher seasonal utilization.',
        'Lower-demand periods may provide opportunities for maintenance and repositioning.',
        'Weather conditions should be considered when planning fleet availability.'
    ],
    'Recommended Action': [
        'Prioritize registered customer retention and service quality.',
        'Plan bike redistribution before evening peak hours.',
        'Schedule suitable maintenance activities during low-demand periods.',
        'Develop separate weekday and weekend operating schedules.',
        'Prepare additional operational capacity for high-demand seasons.',
        'Review fleet utilization and maintenance requirements during lower-demand seasons.',
        'Incorporate weather information into demand planning.'
    ]
})

business_insights

,Finding,Business Implication,Recommended Action
0,Registered customers contribute 81.17% of rent...,Registered customers are the primary contribut...,Prioritize registered customer retention and s...
1,Peak demand occurs at 17:00.,Evening operations require adequate bike avail...,Plan bike redistribution before evening peak h...
2,Lowest demand occurs at 04:00.,"Overnight demand is relatively low, potentiall...",Schedule suitable maintenance activities durin...
3,Weekend demand is 95.89% of weekday demand.,Weekday and weekend demand profiles require di...,Develop separate weekday and weekend operating...
4,Fall has the highest average seasonal demand.,Fleet availability should account for higher s...,Prepare additional operational capacity for hi...
5,Spring has the lowest average seasonal demand.,Lower-demand periods may provide opportunities...,Review fleet utilization and maintenance requi...
6,Light rain/snow demand is 45.54% lower than cl...,Weather conditions should be considered when p...,Incorporate weather information into demand pl...


### Analytical Limitations

1. The dataset covers only 2011 and 2012.
2. Observed associations do not establish causation.
3. Weather category 4 contains only three observations and is excluded from comparative weather conclusions.
4. Rental volume does not represent unique customer counts.
5. Aggregate averages may hide variation across individual dates and locations.
6. The dataset does not provide revenue, operating costs, fleet availability or maintenance records.

## 6.1 Database Schema Preparation

Prepare the transformed analytical dataset for PostgreSQL integration.

The objective is to inspect column names, data types and analytical requirements before designing the database schema.

In [18]:
schema_inventory = pd.DataFrame({
    'Column': business_data.columns,
    'Pandas_Dtype': business_data.dtypes.astype(str).values,
    'Null_Count': business_data.isnull().sum().values,
    'Unique_Values': business_data.nunique().values
})

schema_inventory

,Column,Pandas_Dtype,Null_Count,Unique_Values
0,instant,int64,0,17379
1,dteday,datetime64[us],0,731
2,season,int64,0,4
3,yr,int64,0,2
4,mnth,int64,0,12
5,hr,int64,0,24
6,holiday,int64,0,2
7,weekday,int64,0,7
8,workingday,int64,0,2
9,weathersit,int64,0,4


In [19]:
casual_registered_ratio = (
    total_casual_rentals / total_registered_rentals
)

print("Casual-to-Registered Ratio:", casual_registered_ratio)

print("Total columns:", len(business_data.columns))
print("Total records:", len(business_data))
print("Duplicate records:", business_data.duplicated().sum())
print("Missing values:", business_data.isnull().sum().sum())

Casual-to-Registered Ratio: 0.23198481513936292
Total columns: 32
Total records: 17379
Duplicate records: 0
Missing values: 0


In [20]:
print("Total columns:", len(business_data.columns))
print("Total records:", len(business_data))
print("Duplicate records:", business_data.duplicated().sum())
print("Missing values:", business_data.isnull().sum().sum())

Total columns: 32
Total records: 17379
Duplicate records: 0
Missing values: 0
